# RK1828 Qwen3-ASR 语音识别 Demo


[Qwen3-ASR](https://github.com/QwenLM/Qwen3-ASR)（通义千问）支持 **52 种语言和方言**，
自动判断语种。拆成两部分，全部跑在 NPU 上：

```
音频(wav) -> [Audio Encoder] 提取音频特征 -> [LLM] 自回归生成文字 -> 识别结果
             encoder.rknn (fp16)            llm.rknn (W4A16 量化)
```

| 步骤 | 内容 |
|---|---|
| 1 | 下载模型和示例音频（可选 HF / HF-Mirror 源） |
| 2 | 解压预编译 demo |
| 3 | 选择音频并试听 |
| 4 | 识别 |

## 0. 环境检查

先确认 RKNN3 运行库（`librknn3_api.so`）和 RK1828 设备都正常。
如失败，请先安装驱动/运行时（`rknn3-rk182x` DEB 包），并用 `sudo rknn-smi info` 确认能看到 RK1828。

In [ ]:
from rknn3 import get_lib, find_devices

_ = get_lib()  # 加载 /usr/lib/librknn3_api.so；失败说明 Runtime 未安装

devices = find_devices()
if not devices:
    raise RuntimeError("未发现 RK1828 设备！请检查: sudo rknn-smi info")
for dev_id, dev_type in devices:
    print(f"发现设备: id={dev_id}  接口={dev_type}")

## 1. 下载模型

本 demo 的模型文件托管在 Hugging Face（`qhddz` 账号）。
**先用下拉框选择下载源**，再运行下一个单元格开始下载：

- **HF-Mirror 国内镜像**：`hf-mirror.com`，国内直连，推荐
- **Hugging Face 官方**：`huggingface.co`，海外或已有代理时用

> 已存在（且非空）的文件会**自动跳过**，重复运行不会重新下载。


示例音频来自其他 ASR 仓库，放在 `datasets/asr/` 共用。

In [ ]:
from nb_download import source_picker

# (HF仓库名, 仓库内路径, 本地路径)
FILES = [
    ("qwen3-asr-rknn3-rk1828", "demo_x86_64.tar.gz", "demos/qwen3_asr/demo_x86_64.tar.gz"),
    ("qwen3-asr-rknn3-rk1828", "mel_128_filters.txt", "demos/qwen3_asr/mel_128_filters.txt"),
    ("qwen3-asr-rknn3-rk1828", "model/encoder.rknn", "demos/qwen3_asr/model/encoder.rknn"),
    ("qwen3-asr-rknn3-rk1828", "model/encoder.weight", "demos/qwen3_asr/model/encoder.weight"),
    ("qwen3-asr-rknn3-rk1828", "model/llm.embed.bin", "demos/qwen3_asr/model/llm.embed.bin"),
    ("qwen3-asr-rknn3-rk1828", "model/llm.rknn", "demos/qwen3_asr/model/llm.rknn"),
    ("qwen3-asr-rknn3-rk1828", "model/llm.tokenizer.gguf", "demos/qwen3_asr/model/llm.tokenizer.gguf"),
    ("qwen3-asr-rknn3-rk1828", "model/llm.weight", "demos/qwen3_asr/model/llm.weight"),
    ("whisper-base-rknn3-rk1828", "model/test_zh.wav", "datasets/asr/test_zh.wav"),
    ("whisper-base-rknn3-rk1828", "model/test_en.wav", "datasets/asr/test_en.wav"),
    ("whisper-base-rknn3-rk1828", "model/long_en.wav", "datasets/asr/long_en.wav"),
    ("sensevoice-small-rknn3-rk1828", "model/zh.wav", "datasets/asr/sensevoice_zh.wav"),
    ("paraformer-rknn3-rk1828", "model/asr_example.wav", "datasets/asr/asr_example.wav"),
    ("paraformer-rknn3-rk1828", "model/asr_example_hotword.wav", "datasets/asr/asr_example_hotword.wav"),
    ("qwen3-forced-aligner-rknn3-rk1828", "audio.wav", "datasets/asr/aligner_demo.wav"),
]

w_source = source_picker()

In [ ]:
from nb_download import ensure_files

ensure_files(FILES, w_source.value)

## 2. 解压预编译 demo

`demo_x86_64.tar.gz` 里是官方流程编译好的 x86_64 C++ demo 和运行时库（`lib/`）。
HF 不允许直接托管可执行文件，所以打包成 tar.gz，解压一次即可（重复运行自动跳过）。

In [ ]:
from nb_download import extract_tarball

extract_tarball("demos/qwen3_asr/demo_x86_64.tar.gz", "demos/qwen3_asr")

In [ ]:
from nb_demo import run_demo, snapshot, new_files, show_images, show_new_images, show_audio

## 3. 选择音频并试听

In [ ]:
import glob
import ipywidgets as widgets
from IPython.display import display

w_wav = widgets.Dropdown(options=sorted(glob.glob("datasets/asr/*.wav")),
                         description="音频:", layout=widgets.Layout(width="60%"))
display(w_wav)
if any(o.endswith('/test_zh.wav') for o in w_wav.options):
    w_wav.value = [o for o in w_wav.options if o.endswith('/test_zh.wav')][0]
print("选好后继续往下运行 ▼")

先听一下这段音频：

In [ ]:
from nb_demo import show_audio

show_audio(w_wav.value)

## 4. 识别

`0xff 0xff` 是设备/会话参数占位。实测一段 8.9 秒中文音频可以完整正确转写。

In [ ]:
import os

DEMO = "demos/qwen3_asr"
wav = os.path.abspath(w_wav.value)
run_demo(["./rknn_qwen3_asr_demo",
          "model/encoder.rknn", "model/encoder.weight",
          "model/llm.rknn", "model/llm.weight",
          "model/llm.tokenizer.gguf", "model/llm.embed.bin",
          "0xff", "0xff", wav], DEMO)

## 小结

- 编码器 + LLM 解耦：音频特征提取和文字生成分别在两个 RKNN 模型里完成。
- 压缩包里还有一个 `rknn_qwen3_asr_demo_online`（流式版），可以试试看。